# C5-neural-networks — Practice p14 — Solution

Each rectangle uses left, right, bottom, and top detectors.  Each second-layer unit ANDs its own four detectors with bias $-(4-0.5)=-3.5$; the final unit ORs the two rectangle indicators with bias $-0.5$.

AND-ing half-planes always gives their intersection, which is convex.  Here $T$ is not convex—for example, it contains $(0,2)$ and $(1.5,6)$ but not their midpoint $(0.75,4)$—so one AND over all eight detectors cannot represent the union; the separate ANDs followed by OR supply the needed non-convexity.

In [ ]:
import numpy as np


def affine_layer(x, W, b):
    return (x[:, None, :] * W[None, :, :]).sum(axis=2) + b


def step_activation(z):
    return (z >= 0).astype(float)


SEED = 20260804
probes = np.array([[4.0, 1.0], [2.0, 5.0], [2.0, 1.0], [4.0, 4.0], [0.0, 0.0], [5.0, 2.0]])
rng = np.random.default_rng(SEED)
pts = rng.uniform([-1, -1], [6, 7], (5000, 2))
W1 = np.array([[1.0, 0.0], [-1.0, 0.0], [0.0, 1.0], [0.0, -1.0],
               [1.0, 0.0], [-1.0, 0.0], [0.0, 1.0], [0.0, -1.0]])
b1 = np.array([0.0, 5.0, 0.0, 2.0, -1.5, 3.5, 0.0, 6.0])
W2 = np.array([[1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0],
               [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0]])
b2 = np.array([-3.5, -3.5])
W3 = np.array([[1.0, 1.0]])
b3 = np.array([-0.5])
probe_by_hand = [1.0, 1.0, 1.0, 0.0, 1.0, 1.0]


def union_labels(points):
    h1 = step_activation(affine_layer(points, W1, b1))
    h2 = step_activation(affine_layer(h1, W2, b2))
    return step_activation(affine_layer(h2, W3, b3))[:, 0]


probe_labels = union_labels(probes)
frac_in = float(union_labels(pts).mean())
probe_labels, frac_in

### Answer check

In [ ]:
assert W1.shape == (8, 2) and W2.shape == (2, 8) and W3.shape == (1, 2)
assert np.allclose(probe_labels, probe_by_hand, atol=1e-12)
assert np.isclose(frac_in, 0.313, atol=1e-12)